In [ ]:
import os
import glob
import sys
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.optim import Adam
from torch_geometric.data import Data
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. Import NEST Library from your local workspace
# ---------------------------------------------------------
NEST_LIB_PATH = r'c:\Users\hp\Downloads\aml (3)\aml (1)\aml\case_study'
if NEST_LIB_PATH not in sys.path:
    sys.path.append(NEST_LIB_PATH)

from nest_aml_lib import (
    ExperimentConfig, 
    FullGAT, 
    supervised_loss, 
    evaluate_tuned
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 2. Graph Construction for Tabular Data
# ---------------------------------------------------------
def load_and_build_graph(data_dir):
    print("Loading IEEE-CIS Data...")
    train_transaction = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    train_identity = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))
    
    df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    
    print("Preprocessing node features...")
    y = df['isFraud'].values
    df = df.fillna(0)
    
    # Label encode categorical columns
    cat_cols = [c for c in df.columns if df[c].dtype == 'object']
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        
    drop_cols = ['isFraud', 'TransactionID', 'TransactionDT']
    X = df.drop(drop_cols, axis=1).values
    
    # Standardize features for the NEST Encoder
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    print("Building Graph Edges (linking shared 'card1')...")
    # To prevent O(N^2) memory issues, we sort and link consecutive transactions 
    # that belong to the same credit card (card1)
    df['orig_idx'] = np.arange(len(df))
    df = df.sort_values(by=['card1', 'TransactionDT'])
    
    src, dst = [], []
    prev_card, prev_idx = None, None
    
    for _, row in df.iterrows():
        curr_card = row['card1']
        curr_idx = int(row['orig_idx'])
        
        # Link consecutive transactions on the same card
        if curr_card == prev_card and prev_idx is not None:
            src.append(prev_idx)
            dst.append(curr_idx)
            # Add reverse edge to make it undirected
            src.append(curr_idx)
            dst.append(prev_idx)
            
        prev_card = curr_card
        prev_idx = curr_idx
        
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    x_tensor = torch.tensor(X, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    # Recreate the PyTorch Geometric Data object expected by NEST
    data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)
    
    print(f"Graph Built: {data.num_nodes} nodes, {data.num_edges} edges, {data.num_node_features} features")
    return data

# ---------------------------------------------------------
# 3. K-Fold Training with Glob Checkpointing
# ---------------------------------------------------------
def run_3fold_cv_nest(data_dir='./data', checkpoint_dir='./nest_checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Force the window size to 1 to keep edges low and prevent OOM
    data = load_and_build_graph(data_dir, card_time_window=1)
    n_nodes = data.num_nodes
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    y_numpy = data.y.numpy()
    indices = np.arange(n_nodes)
    
    # Initialize your paper's configuration
    cfg = ExperimentConfig()
    
    # ---------------------------------------------------------
    # USE GLOB TO FIND EXISTING CHECKPOINTS
    # ---------------------------------------------------------
    checkpoint_pattern = os.path.join(checkpoint_dir, 'nest_ieee_fold_*.pt')
    found_checkpoints = glob.glob(checkpoint_pattern)
    print(f"\nFound {len(found_checkpoints)} existing checkpoints using glob: {found_checkpoints}\n")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, y_numpy)):
        fold_num = fold + 1
        print(f"{'='*20} Fold {fold_num} {'='*20}")
        
        fold_checkpoint_path = os.path.join(checkpoint_dir, f'nest_ieee_fold_{fold_num}.pt')
        
        # Instantiate your FullGAT model
        model = FullGAT(in_dim=data.num_node_features, cfg=cfg).to(DEVICE)
        
        # Generate PyTorch Boolean Masks for evaluating sub-graphs
        train_mask = torch.zeros(n_nodes, dtype=torch.bool)
        train_mask[train_idx] = True
        
        val_mask = torch.zeros(n_nodes, dtype=torch.bool)
        val_mask[val_idx] = True
        
        # Check if fold already trained
        if fold_checkpoint_path in found_checkpoints:
            print(f"--> Checkpoint found. Loading model from {fold_checkpoint_path}...")
            model.load_state_dict(torch.load(fold_checkpoint_path, map_location=DEVICE))
            
            metrics = evaluate_tuned(model, data, train_mask, val_mask, DEVICE, cfg)
            print(f"--> Fold {fold_num} (Loaded) AUC: {metrics['auc']:.5f} | F1: {metrics['f1']:.5f}\n")
            continue
            
        print(f"--> No checkpoint found for Fold {fold_num}. Training from scratch...")
        optimizer = Adam(model.parameters(), lr=cfg.lr)
        
        # Training Loop
        for epoch in range(cfg.sup_epochs):
            model.train()
            optimizer.zero_grad()
            
            # Forward pass through SAGEGATEncoder & ClassHead
            logits, _ = model(data.x.to(DEVICE), data.edge_index.to(DEVICE))
            
            # Isolate training nodes
            logits_train = logits[train_mask]
            y_train = data.y[train_mask].to(DEVICE)
            
            # Uses focal_loss natively as defined in your config
            loss = supervised_loss(logits_train, y_train, DEVICE, cfg)
            loss.backward()
            optimizer.step()
            
            if (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1}/{cfg.sup_epochs} - Loss: {loss.item():.4f}")
                
        # Save PyTorch checkpoint
        torch.save(model.state_dict(), fold_checkpoint_path)
        print(f"--> Saved checkpoint to {fold_checkpoint_path}")
        
        # Evaluate validation fold using your internal library logic
        metrics = evaluate_tuned(model, data, train_mask, val_mask, DEVICE, cfg)
        print(f"--> Fold {fold_num} (Trained) AUC: {metrics['auc']:.5f} | F1: {metrics['f1']:.5f}\n")

if __name__ == '__main__':
    # Define directories
    DATA_DIRECTORY = r'C:\Users\hp\Downloads\ieee_cis' # Change to your actual IEEE-CIS path
    CHECKPOINT_DIRECTORY = r'C:\Users\hp\Downloads\aml (3)\aml (1)\checkpoints'
    
    run_3fold_cv_nest(data_dir=DATA_DIRECTORY, checkpoint_dir=CHECKPOINT_DIRECTORY)

In [ ]:
!pip install torch_geometric

In [ ]:
import os
import glob
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from dataclasses import dataclass
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, precision_recall_curve
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings

warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# Model definitions -- copied VERBATIM from notebook1.ipynb (the source of
# truth for your Elliptic/EP-FedProto results) instead of importing from a
# separate nest_aml_lib dataset. This removes any risk of the IEEE-CIS run
# silently using a diverged/older copy of ExperimentConfig or FullGAT.
# Only the fields actually used by this script's FullGAT/CV loop are kept;
# the full dataclass and forward_od/trunc_dim machinery are kept intact so
# this stays a drop-in match with notebook1's version.
# =============================================================================

@dataclass
class ExperimentConfig:
    # Architecture
    hidden:       int   = 128
    emb_dim:      int   = 64
    heads:        int   = 4
    dropout:      float = 0.4
    head_dropout: float = 0.3

    # Federation (unused by this centralized IEEE-CIS script, kept for parity)
    n_clients:  int   = 4
    seeds:      tuple = (42, 123, 7, 456, 789)
    test_ratio: float = 0.3

    use_dev_features: bool = True

    # Training schedule (unused here -- this script has its own epoch loop)
    global_rounds:        int   = 100
    head_finetune_rounds: int   = 20
    ssl_pretrain_rounds:  int   = 6
    ssl_epochs:           int   = 20
    sup_epochs:           int   = 25
    lr:                   float = 0.005
    lr_min:               float = 0.0005

    mu_encoder: float = 0.01
    mu_head:    float = 0.0

    tau:            float = 0.7
    ssl_edge_floor: int   = 15
    feat_drop:      float = 0.3
    edge_drop:      float = 0.2

    lam_max:              float = 0.6
    lam_warmup_rounds:    int   = 2
    ema_momentum:         float = 0.85
    raw_blend:            float = 0.2
    use_degree_weighting: bool  = True

    use_ssl:     bool = True
    use_protos:  bool = True
    use_fedprox: bool = True

    use_contrib_agg:  bool  = True
    contrib_floor:    float = 0.1
    use_grad_guard:   bool  = True
    anomaly_thresh:   float = -0.1
    use_saliency:     bool  = True
    saliency_top_k:   int   = 20
    use_calibration:  bool  = True
    ece_bins:         int   = 15

    use_focal_loss:   bool  = True
    focal_gamma:      float = 1.0

    use_label_prop:   bool  = False
    lp_alpha:         float = 0.9
    lp_steps:         int   = 1

    use_fedper:       bool  = True

    cosine_T0:        int   = 50

    baseline_rounds:       int = 50
    baseline_local_epochs: int = 25
    centralized_epochs:    int = 150


class SAGEGATEncoder(nn.Module):
    """
    Encoder: SAGEConv x2 (skip connections) -> GATv2Conv (query-dependent attention).
    Verbatim from notebook1.ipynb.
    """

    def __init__(self, in_dim: int, cfg: ExperimentConfig):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.conv1 = SAGEConv(in_dim, h)
        self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False)
        self.skip1 = nn.Linear(in_dim, h, bias=False)
        self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1   = nn.BatchNorm1d(h)
        self.bn2   = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(
            nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, e)
        )
        self.raw_projector = nn.Linear(in_dim, e, bias=False)

    def _sage_layers(self, x, edge_index):
        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        return h2

    def forward(self, x, edge_index):
        h2 = self._sage_layers(x, edge_index)
        return self.conv3(h2, edge_index)

    def forward_with_attention(self, x, edge_index):
        h2 = self._sage_layers(x, edge_index)
        emb, (att_edge_index, att_weights) = self.conv3(
            h2, edge_index, return_attention_weights=True
        )
        return emb, att_edge_index, att_weights

    def encode_with_proj(self, x, edge_index):
        z = self.forward(x, edge_index)
        return z, self.proj_head(z)

    def forward_od(self, x, edge_index, width_ratio: float):
        """Ordered Dropout (FjORD) forward pass -- unused by this script but
        kept for parity with notebook1's encoder."""
        def _od_mask(h):
            keep = max(1, round(h.shape[1] * width_ratio))
            mask = torch.zeros_like(h)
            mask[:, :keep] = 1.0
            return h * mask

        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = _od_mask(h1)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = _od_mask(h2)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        emb = self.conv3(h2, edge_index)
        emb = _od_mask(emb)
        return emb


class ClassHead(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig, n_classes: int = 2):
        super().__init__()
        h1 = max(128, in_dim * 2)
        h2 = h1 // 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout / 2),
            nn.Linear(h2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class FullGAT(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg)
        self.head    = ClassHead(cfg.emb_dim, cfg)

    def forward(self, x, edge_index, trunc_dim: int = None):
        """trunc_dim: EP-FedProto v3 fixed-tier baseline hook -- unused by this
        script (always called with trunc_dim=None), kept for parity with
        notebook1's FullGAT.forward signature."""
        emb = self.encoder(x, edge_index)
        if trunc_dim is not None:
            mask = torch.zeros_like(emb)
            mask[:, :trunc_dim] = 1.0
            emb = emb * mask
        return self.head(emb), emb


def load_and_build_graph(data_dir, card_time_window: int = 5):
    print("Loading IEEE-CIS Data...")
    train_transaction = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    train_identity = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))

    df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    y = df['isFraud'].values

    # -----------------------------------------------------
    # FIX 1: SEPARATE NUMERIC / CATEGORICAL MISSING-VALUE HANDLING
    # -----------------------------------------------------
    # BUG (original): fillna(-999) was applied to the WHOLE dataframe, then only
    # categorical columns were re-mapped via frequency encoding. Every numeric
    # column (V1-V339, D1-D15, id_01..id_38, etc. -- many 60-95% missing in
    # IEEE-CIS) kept the literal value -999 and was fed straight into
    # StandardScaler. That sentinel dominates the column mean/std, so after
    # scaling almost all the REAL values collapse into a narrow band near the
    # same normalized value -- the signal in most engineered features gets
    # destroyed before the model ever sees it. This is very likely why F1 was bad.
    print("Preprocessing features (frequency encoding + numeric imputation, no -999 into scaler)...")

    cat_cols = list(df.select_dtypes(include=['object']).columns)
    id_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6',
               'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']
    cat_cols = list(set(cat_cols + [c for c in id_cols if c in df.columns]))

    drop_cols = ['isFraud', 'TransactionID', 'TransactionDT']
    numeric_cols = [c for c in df.columns
                    if c not in drop_cols and c not in cat_cols
                    and pd.api.types.is_numeric_dtype(df[c])]

    # Categorical: fill NaN with a distinct placeholder BEFORE frequency
    # encoding (fine -- "missing" becomes its own category), then map to
    # frequency. This part of the original logic was correct.
    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        freq = df[col].value_counts(normalize=True)
        df[col] = df[col].map(freq).fillna(0)

    # Numeric: add a missing-indicator (very informative in IEEE-CIS -- missingness
    # patterns in the V-columns correlate strongly with fraud), then impute with
    # the column MEDIAN (robust to outliers) instead of a sentinel. This keeps
    # the true value distribution intact for StandardScaler.
    missing_indicator_cols = []
    for col in numeric_cols:
        n_missing = df[col].isna().sum()
        if n_missing > 0:
            ind_col = f'{col}__isna'
            df[ind_col] = df[col].isna().astype(np.float32)
            missing_indicator_cols.append(ind_col)
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)

    # FIX: log1p on the heavily right-skewed amount column -- standard trick,
    # keeps large transactions from dominating the scaled feature space.
    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    feature_cols = numeric_cols + cat_cols + missing_indicator_cols
    X = df[feature_cols].values.astype(np.float32)

    # NOTE: fit on the full feature matrix (train+val nodes, no labels) --
    # acceptable here since this is a *transductive* setting (all node features
    # are visible at train time, only labels are held out per fold), same as
    # your other NEST/EP-FedProto notebooks. If you want a stricter inductive
    # eval, refit the scaler inside the fold loop on train_idx rows only.
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # -----------------------------------------------------
    # FIX 2: RICHER GRAPH EDGES
    # -----------------------------------------------------
    # BUG (original): edges were a linear chain within each card1 group, sorted
    # by time (idx[i] <-> idx[i+1] only). Two problems: (a) most of the graph's
    # message-passing signal is thrown away -- a transaction only ever hears
    # from its immediate temporal neighbor, not the rest of the group; (b) many
    # card1 values are near-unique, leaving large numbers of isolated nodes.
    # FIX: connect each transaction to its `card_time_window` nearest temporal
    # neighbors within the same card1 group (a small sliding window), which
    # gives denser, still time-respecting local neighborhoods without an
    # explosion of edges from fully cliquing large groups.
    print(f"Building graph edges (card1 groups, time-window={card_time_window})...")
    df['orig_idx'] = np.arange(len(df))
    src, dst = [], []

    df_sorted = df.sort_values(by=['card1', 'TransactionDT'])
    for _, group in df_sorted.groupby('card1'):
        idxs = group['orig_idx'].values
        n = len(idxs)
        if n < 2:
            continue
        for offset in range(1, min(card_time_window, n - 1) + 1):
            a = idxs[:-offset]
            b = idxs[offset:]
            src.extend(a); dst.extend(b)
            src.extend(b); dst.extend(a)

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    data = Data(x=torch.tensor(X, dtype=torch.float), edge_index=edge_index,
                y=torch.tensor(y, dtype=torch.long))
    n_isolated = data.num_nodes - torch.unique(edge_index).numel()
    print(f"Graph Built: {data.num_nodes:,} nodes | {data.num_edges:,} edges | "
          f"{data.num_node_features} features | ~{n_isolated:,} isolated nodes")
    return data


def tune_threshold_on_train(model, projector, data, train_mask):
    model.eval()
    projector.eval()
    with torch.no_grad():
        # Evaluate in 16-bit precision to save memory
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            x_proj = projector(data.x)
            logits, _ = model(x_proj, data.edge_index)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

    y_train = data.y[train_mask].cpu().numpy()
    probs_train = probs[train_mask.cpu().numpy()]
    if len(np.unique(y_train)) < 2:
        return 0.5, probs
    precisions, recalls, thresholds = precision_recall_curve(y_train, probs_train)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores[:-1]) if len(thresholds) else 0
    best_thresh = float(np.clip(thresholds[best_idx], 0.05, 0.95)) if len(thresholds) else 0.5
    return best_thresh, probs


def evaluate_with_threshold(probs, data, val_mask, thresh):
    y_val = data.y[val_mask].cpu().numpy()
    probs_val = probs[val_mask.cpu().numpy()]
    preds_val = (probs_val >= thresh).astype(int)
    return {
        'auc': roc_auc_score(y_val, probs_val),
        'f1': f1_score(y_val, preds_val, zero_division=0),
        'prec': precision_score(y_val, preds_val, zero_division=0),
        'rec': recall_score(y_val, preds_val, zero_division=0),
        'best_thresh': thresh,
    }


def run_3fold_cv_nest(data_dir, checkpoint_dir, epochs=500):
    os.makedirs(checkpoint_dir, exist_ok=True)
    data = load_and_build_graph(data_dir, card_time_window=1)
    
    # Move the entire graph to GPU once to avoid memory fragmentation
    data = data.to(DEVICE)
    
    n_nodes = data.num_nodes
    y_numpy = data.y.cpu().numpy()

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cfg = ExperimentConfig()
    cfg.hidden = 128
    cfg.emb_dim = 64

    pos_weight_raw = (y_numpy == 0).sum() / max((y_numpy == 1).sum(), 1)
    pos_weight = float(np.clip(pos_weight_raw, 0.1, 10.0))
    print(f"Raw pos_weight={pos_weight_raw:.2f} -> clamped to {pos_weight:.2f}")
    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    found_checkpoints = glob.glob(os.path.join(checkpoint_dir, 'nest_ieee_v5_fold_*.pt'))
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(n_nodes), y_numpy)):
        fold_num = fold + 1
        print(f"\n{'='*25} Fold {fold_num} {'='*25}")
        fold_checkpoint_path = os.path.join(checkpoint_dir, f'nest_ieee_v5_fold_{fold_num}.pt')

        projector = nn.Linear(data.num_node_features, 128).to(DEVICE)
        model = FullGAT(in_dim=128, cfg=cfg).to(DEVICE)
        
        train_mask = torch.zeros(n_nodes, dtype=torch.bool).to(DEVICE)
        train_mask[train_idx] = True
        val_mask = torch.zeros(n_nodes, dtype=torch.bool).to(DEVICE)
        val_mask[val_idx] = True

        if fold_checkpoint_path in found_checkpoints:
            state = torch.load(fold_checkpoint_path, map_location=DEVICE)
            model.load_state_dict(state['model'])
            projector.load_state_dict(state['projector'])
            thresh, probs = tune_threshold_on_train(model, projector, data, train_mask)
            metrics = evaluate_with_threshold(probs, data, val_mask, thresh)
            print(f"--> Fold {fold_num} (Loaded): AUC: {metrics['auc']:.4f} | F1: {metrics['f1']:.4f} | thresh={thresh:.3f}")
            fold_metrics.append(metrics)
            continue

        print(f"--> Training Fold {fold_num} for {epochs} epochs (AMP enabled)...")
        optimizer = AdamW(list(model.parameters()) + list(projector.parameters()), lr=0.005, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-4)
        
        # -----------------------------------------------------
        # AMP GRADIENT SCALER (Cuts VRAM usage by 50%)
        # -----------------------------------------------------
        scaler_amp = torch.cuda.amp.GradScaler()

        best_val_f1 = 0.0
        best_metrics = None

        for epoch in range(1, epochs + 1):
            model.train()
            projector.train()
            
            # set_to_none=True slightly reduces VRAM
            optimizer.zero_grad(set_to_none=True) 

            # Run forward pass in float16
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                x_proj = projector(data.x)
                logits, _ = model(x_proj, data.edge_index)
                loss = criterion(logits[train_mask], data.y[train_mask])

            # Scale gradients and step
            scaler_amp.scale(loss).backward()
            
            # Unscale before clipping to ensure correct gradient norms
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            scaler_amp.step(optimizer)
            scaler_amp.update()
            scheduler.step()

            if epoch % 50 == 0 or epoch == epochs:
                thresh, probs = tune_threshold_on_train(model, projector, data, train_mask)
                metrics = evaluate_with_threshold(probs, data, val_mask, thresh)
                print(f"  Epoch {epoch:03d}/{epochs} | Loss: {loss.item():.4f} | "
                      f"Val AUC: {metrics['auc']:.4f} | Val F1: {metrics['f1']:.4f} | "
                      f"thresh={thresh:.3f}")

                if metrics['f1'] > best_val_f1:
                    best_val_f1 = metrics['f1']
                    best_metrics = metrics
                    torch.save({'model': model.state_dict(), 'projector': projector.state_dict()}, fold_checkpoint_path)

        print(f"--> Best Checkpoint saved for Fold {fold_num}")
        print(f"--> Final Peak F1: {best_val_f1:.4f}")
        fold_metrics.append(best_metrics)

    if fold_metrics:
        f1s = [m['f1'] for m in fold_metrics if m]
        aucs = [m['auc'] for m in fold_metrics if m]
        print(f"\n{'='*25} 3-Fold CV Summary {'='*25}")
        print(f"F1  = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
        print(f"AUC = {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}")

    return fold_metrics


if __name__ == '__main__':
    DATA_DIRECTORY = '/kaggle/input/competitions/ieee-fraud-detection'
    CHECKPOINT_DIRECTORY = '/kaggle/working'

    run_3fold_cv_nest(data_dir=DATA_DIRECTORY, checkpoint_dir=CHECKPOINT_DIRECTORY, epochs=500)

In [ ]:
import os
import glob
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from dataclasses import dataclass
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, precision_recall_curve
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings

warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# ExperimentConfig -- verbatim from notebook1
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden:       int   = 128
    emb_dim:      int   = 64
    heads:        int   = 4
    dropout:      float = 0.4
    head_dropout: float = 0.3

    n_clients:  int   = 4
    seeds:      tuple = (42, 123, 7, 456, 789)
    test_ratio: float = 0.3
    use_dev_features: bool = True

    global_rounds:        int   = 100
    head_finetune_rounds: int   = 20
    ssl_pretrain_rounds:  int   = 6
    ssl_epochs:           int   = 20
    sup_epochs:           int   = 25
    lr:                   float = 0.005
    lr_min:               float = 0.0005

    mu_encoder: float = 0.01
    mu_head:    float = 0.0
    tau:            float = 0.7
    ssl_edge_floor: int   = 15
    feat_drop:      float = 0.3
    edge_drop:      float = 0.2

    lam_max:              float = 0.6
    lam_warmup_rounds:    int   = 2
    ema_momentum:         float = 0.85
    raw_blend:            float = 0.2
    use_degree_weighting: bool  = True

    use_ssl:     bool = True
    use_protos:  bool = True
    use_fedprox: bool = True

    use_contrib_agg:  bool  = True
    contrib_floor:    float = 0.1
    use_grad_guard:   bool  = True
    anomaly_thresh:   float = -0.1
    use_saliency:     bool  = True
    saliency_top_k:   int   = 20
    use_calibration:  bool  = True
    ece_bins:         int   = 15

    use_focal_loss:   bool  = True
    focal_gamma:      float = 1.0

    use_label_prop:   bool  = False
    lp_alpha:         float = 0.9
    lp_steps:         int   = 1

    use_fedper:       bool  = True
    cosine_T0:        int   = 50

    baseline_rounds:       int = 50
    baseline_local_epochs: int = 25
    centralized_epochs:    int = 150


# =============================================================================
# MODIFIED SAGEGATEncoder -- GATv2Conv now accepts edge_attr (time deltas)
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig, edge_dim: int = None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.conv1 = SAGEConv(in_dim, h)
        self.conv2 = SAGEConv(h, h)
        # FIX 1: pass edge_dim so GATv2Conv can use time-delta edge attributes
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False,
                               edge_dim=edge_dim)
        self.skip1 = nn.Linear(in_dim, h, bias=False)
        self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1   = nn.BatchNorm1d(h)
        self.bn2   = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(
            nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, e)
        )
        self.raw_projector = nn.Linear(in_dim, e, bias=False)

    def _sage_layers(self, x, edge_index):
        h1 = self.conv1(x, edge_index) + self.skip1(x)
        h1 = F.relu(self.bn1(h1))
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.conv2(h1, edge_index) + self.skip2(h1)
        h2 = F.relu(self.bn2(h2))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        return h2

    def forward(self, x, edge_index, edge_attr=None):
        h2 = self._sage_layers(x, edge_index)
        return self.conv3(h2, edge_index, edge_attr=edge_attr)


class ClassHead(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig, n_classes: int = 2):
        super().__init__()
        h1 = max(128, in_dim * 2)
        h2 = h1 // 2
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(cfg.head_dropout / 2),
            nn.Linear(h2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class FullGAT(nn.Module):
    def __init__(self, in_dim: int, cfg: ExperimentConfig, edge_dim: int = None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head    = ClassHead(cfg.emb_dim, cfg)

    def forward(self, x, edge_index, edge_attr=None, trunc_dim: int = None):
        emb = self.encoder(x, edge_index, edge_attr=edge_attr)
        if trunc_dim is not None:
            mask = torch.zeros_like(emb)
            mask[:, :trunc_dim] = 1.0
            emb = emb * mask
        return self.head(emb), emb


# =============================================================================
# Data Loading with all 3 fixes
# =============================================================================
def load_and_build_graph(data_dir, card_time_window: int = 1):
    print("Loading IEEE-CIS Data...")
    train_transaction = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    train_identity = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))

    df = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    y = df['isFraud'].values

    print("Preprocessing features...")

    cat_cols = list(df.select_dtypes(include=['object']).columns)
    id_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6',
               'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']
    cat_cols = list(set(cat_cols + [c for c in id_cols if c in df.columns]))

    drop_cols = ['isFraud', 'TransactionID']
    numeric_cols = [c for c in df.columns
                    if c not in drop_cols and c not in cat_cols
                    and pd.api.types.is_numeric_dtype(df[c])]

    # Frequency encode categoricals
    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        freq = df[col].value_counts(normalize=True)
        df[col] = df[col].map(freq).fillna(0)

    # Numeric: missing indicators + median imputation
    missing_indicator_cols = []
    for col in numeric_cols:
        n_missing = df[col].isna().sum()
        if n_missing > 0:
            ind_col = f'{col}__isna'
            df[ind_col] = df[col].isna().astype(np.float32)
            missing_indicator_cols.append(ind_col)
            df[col] = df[col].fillna(df[col].median())

    # Log-transform skewed amount
    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    # ------------------------------------------------------------------
    # FIX 2: Per-card aggregation features (give the GNN statistical context)
    # ------------------------------------------------------------------
    print("Engineering per-card aggregation features...")
    agg_new_cols = []
    for agg_col in ['TransactionAmt', 'TransactionDT']:
        if agg_col not in df.columns:
            continue
        grp = df.groupby('card1')[agg_col]
        
        mean_col = f'{agg_col}_card1_mean'
        std_col  = f'{agg_col}_card1_std'
        cnt_col  = f'{agg_col}_card1_count'
        
        df[mean_col] = grp.transform('mean')
        df[std_col]  = grp.transform('std').fillna(0)
        df[cnt_col]  = grp.transform('count')
        agg_new_cols.extend([mean_col, std_col, cnt_col])

    # Deviation of this transaction from card average
    if 'TransactionAmt' in df.columns and 'TransactionAmt_card1_mean' in df.columns:
        df['Amt_card1_deviation'] = df['TransactionAmt'] - df['TransactionAmt_card1_mean']
        agg_new_cols.append('Amt_card1_deviation')

    # Keep TransactionDT out of the feature matrix (it's used for edges)
    feature_cols = [c for c in numeric_cols if c != 'TransactionDT'] + cat_cols + missing_indicator_cols + agg_new_cols

    X = df[feature_cols].values.astype(np.float32)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # ------------------------------------------------------------------
    # Build graph edges + FIX 1: time-delta edge attributes
    # ------------------------------------------------------------------
    print(f"Building graph edges (card1 groups, time-window={card_time_window}) with time-delta edge attrs...")
    df['orig_idx'] = np.arange(len(df))
    src, dst, edge_dt = [], [], []

    df_sorted = df.sort_values(by=['card1', 'TransactionDT'])
    for _, group in df_sorted.groupby('card1'):
        idxs = group['orig_idx'].values
        times = group['TransactionDT'].values
        n = len(idxs)
        if n < 2:
            continue
        for offset in range(1, min(card_time_window, n - 1) + 1):
            a = idxs[:-offset]
            b = idxs[offset:]
            dt = np.abs(times[offset:] - times[:-offset]).astype(np.float32)
            # Forward edges
            src.extend(a); dst.extend(b); edge_dt.extend(dt)
            # Reverse edges (same time delta)
            src.extend(b); dst.extend(a); edge_dt.extend(dt)

    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # Normalize time deltas with log1p (raw values span seconds to months)
    edge_dt_arr = np.array(edge_dt, dtype=np.float32)
    edge_attr = torch.tensor(np.log1p(edge_dt_arr), dtype=torch.float).unsqueeze(-1)  # shape [E, 1]

    data = Data(
        x=torch.tensor(X, dtype=torch.float),
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor(y, dtype=torch.long)
    )
    n_isolated = data.num_nodes - torch.unique(edge_index).numel()
    print(f"Graph Built: {data.num_nodes:,} nodes | {data.num_edges:,} edges | "
          f"{data.num_node_features} features | edge_attr dim=1 | ~{n_isolated:,} isolated nodes")
    return data


# =============================================================================
# Threshold tuning & evaluation (unchanged logic)
# =============================================================================
def tune_threshold_on_train(model, projector, data, train_mask):
    model.eval()
    projector.eval()
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            x_proj = projector(data.x)
            logits, _ = model(x_proj, data.edge_index, edge_attr=data.edge_attr)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

    y_train = data.y[train_mask].cpu().numpy()
    probs_train = probs[train_mask.cpu().numpy()]
    if len(np.unique(y_train)) < 2:
        return 0.5, probs
    precisions, recalls, thresholds = precision_recall_curve(y_train, probs_train)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores[:-1]) if len(thresholds) else 0
    best_thresh = float(np.clip(thresholds[best_idx], 0.05, 0.95)) if len(thresholds) else 0.5
    return best_thresh, probs


def evaluate_with_threshold(probs, data, val_mask, thresh):
    y_val = data.y[val_mask].cpu().numpy()
    probs_val = probs[val_mask.cpu().numpy()]
    preds_val = (probs_val >= thresh).astype(int)
    return {
        'auc': roc_auc_score(y_val, probs_val),
        'f1': f1_score(y_val, preds_val, zero_division=0),
        'prec': precision_score(y_val, preds_val, zero_division=0),
        'rec': recall_score(y_val, preds_val, zero_division=0),
        'best_thresh': thresh,
    }


# =============================================================================
# 3-Fold CV with all fixes + 1500 epochs
# =============================================================================
def run_3fold_cv_nest(data_dir, checkpoint_dir, epochs=1500):
    os.makedirs(checkpoint_dir, exist_ok=True)
    data = load_and_build_graph(data_dir, card_time_window=1)
    data = data.to(DEVICE)

    n_nodes = data.num_nodes
    y_numpy = data.y.cpu().numpy()

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cfg = ExperimentConfig()
    cfg.hidden = 128
    cfg.emb_dim = 64

    pos_weight_raw = (y_numpy == 0).sum() / max((y_numpy == 1).sum(), 1)
    pos_weight = float(np.clip(pos_weight_raw, 0.1, 10.0))
    print(f"Raw pos_weight={pos_weight_raw:.2f} -> clamped to {pos_weight:.2f}")
    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    found_checkpoints = glob.glob(os.path.join(checkpoint_dir, 'nest_ieee_v6_fold_*.pt'))
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(n_nodes), y_numpy)):
        fold_num = fold + 1
        print(f"\n{'='*25} Fold {fold_num} {'='*25}")
        fold_checkpoint_path = os.path.join(checkpoint_dir, f'nest_ieee_v6_fold_{fold_num}.pt')

        projector = nn.Linear(data.num_node_features, 128).to(DEVICE)
        # edge_dim=1 tells GATv2Conv to use the time-delta edge attribute
        model = FullGAT(in_dim=128, cfg=cfg, edge_dim=1).to(DEVICE)

        train_mask = torch.zeros(n_nodes, dtype=torch.bool).to(DEVICE)
        train_mask[train_idx] = True
        val_mask = torch.zeros(n_nodes, dtype=torch.bool).to(DEVICE)
        val_mask[val_idx] = True

        if fold_checkpoint_path in found_checkpoints:
            state = torch.load(fold_checkpoint_path, map_location=DEVICE)
            model.load_state_dict(state['model'])
            projector.load_state_dict(state['projector'])
            thresh, probs = tune_threshold_on_train(model, projector, data, train_mask)
            metrics = evaluate_with_threshold(probs, data, val_mask, thresh)
            print(f"--> Fold {fold_num} (Loaded): AUC: {metrics['auc']:.4f} | F1: {metrics['f1']:.4f}")
            fold_metrics.append(metrics)
            continue

        print(f"--> Training Fold {fold_num} for {epochs} epochs (AMP enabled)...")
        optimizer = AdamW(list(model.parameters()) + list(projector.parameters()),
                         lr=0.005, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
        scaler_amp = torch.cuda.amp.GradScaler()

        best_val_f1 = 0.0
        best_metrics = None
        patience_counter = 0
        patience_limit = 200  # Early stop if no improvement for 200 epochs

        for epoch in range(1, epochs + 1):
            model.train()
            projector.train()
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                x_proj = projector(data.x)
                logits, _ = model(x_proj, data.edge_index, edge_attr=data.edge_attr)
                loss = criterion(logits[train_mask], data.y[train_mask])

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            scheduler.step()

            if epoch % 100 == 0 or epoch == epochs:
                thresh, probs = tune_threshold_on_train(model, projector, data, train_mask)
                metrics = evaluate_with_threshold(probs, data, val_mask, thresh)
                print(f"  Epoch {epoch:04d}/{epochs} | Loss: {loss.item():.4f} | "
                      f"Val AUC: {metrics['auc']:.4f} | Val F1: {metrics['f1']:.4f} | "
                      f"Prec: {metrics['prec']:.4f} | Rec: {metrics['rec']:.4f} | "
                      f"thresh={thresh:.3f}")

                if metrics['f1'] > best_val_f1:
                    best_val_f1 = metrics['f1']
                    best_metrics = metrics
                    patience_counter = 0
                    torch.save({
                        'model': model.state_dict(),
                        'projector': projector.state_dict()
                    }, fold_checkpoint_path)
                else:
                    patience_counter += 100

                if patience_counter >= patience_limit:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {patience_limit} epochs)")
                    break

        print(f"--> Best Checkpoint saved for Fold {fold_num}")
        print(f"--> Final Peak F1: {best_val_f1:.4f}")
        fold_metrics.append(best_metrics)

    if fold_metrics:
        f1s = [m['f1'] for m in fold_metrics if m]
        aucs = [m['auc'] for m in fold_metrics if m]
        print(f"\n{'='*25} 3-Fold CV Summary {'='*25}")
        print(f"F1  = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
        print(f"AUC = {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}")

    return fold_metrics


if __name__ == '__main__':
    DATA_DIRECTORY = '/kaggle/input/competitions/ieee-fraud-detection'
    CHECKPOINT_DIRECTORY = '/kaggle/working'

    run_3fold_cv_nest(data_dir=DATA_DIRECTORY, checkpoint_dir=CHECKPOINT_DIRECTORY, epochs=1500)

In [ ]:
import os, glob, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# Configuration
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    global_rounds: int = 100;  head_finetune_rounds: int = 20
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 25
    lr: float = 0.005;  lr_min: float = 0.0005
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    lam_max: float = 0.6;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# =============================================================================
# Data
# =============================================================================
def load_and_build_graph(data_dir, card_time_window=1):
    print("Loading IEEE-CIS Data...")
    txn = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    idn = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))
    df = pd.merge(txn, idn, on='TransactionID', how='left')
    y = df['isFraud'].values

    cat_cols = list(set(list(df.select_dtypes(include=['object']).columns) +
                        [c for c in ['card1','card2','card3','card4','card5','card6',
                                     'addr1','addr2','P_emaildomain','R_emaildomain'] if c in df.columns]))
    drop_cols = ['isFraud', 'TransactionID']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'; df[ic] = df[col].isna().astype(np.float32); miss_cols.append(ic)
            df[col] = df[col].fillna(df[col].median())

    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    agg_cols = []
    for ac in ['TransactionAmt', 'TransactionDT']:
        if ac in df.columns:
            grp = df.groupby('card1')[ac]
            mc, sc, cc = f'{ac}_c1_mean', f'{ac}_c1_std', f'{ac}_c1_count'
            df[mc] = grp.transform('mean'); df[sc] = grp.transform('std').fillna(0); df[cc] = grp.transform('count')
            agg_cols.extend([mc, sc, cc])
    if 'TransactionAmt' in df.columns and 'TransactionAmt_c1_mean' in df.columns:
        df['Amt_c1_dev'] = df['TransactionAmt'] - df['TransactionAmt_c1_mean']; agg_cols.append('Amt_c1_dev')

    feat_cols = [c for c in num_cols if c != 'TransactionDT'] + cat_cols + miss_cols + agg_cols
    X = np.nan_to_num(StandardScaler().fit_transform(df[feat_cols].values.astype(np.float32)))

    df['orig_idx'] = np.arange(len(df))
    src, dst, edge_dt = [], [], []
    for _, g in df.sort_values(['card1','TransactionDT']).groupby('card1'):
        idxs, times = g['orig_idx'].values, g['TransactionDT'].values
        if len(idxs) < 2: continue
        for off in range(1, min(card_time_window, len(idxs)-1)+1):
            a, b, dt = idxs[:-off], idxs[off:], np.abs(times[off:]-times[:-off]).astype(np.float32)
            src.extend(a); dst.extend(b); edge_dt.extend(dt)
            src.extend(b); dst.extend(a); edge_dt.extend(dt)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=torch.tensor([src,dst], dtype=torch.long),
                edge_attr=torch.tensor(np.log1p(np.array(edge_dt, dtype=np.float32))).unsqueeze(-1),
                y=torch.tensor(y, dtype=torch.long),
                timestep=torch.tensor(df['TransactionDT'].values, dtype=torch.long))
    print(f"Graph: {data.num_nodes} nodes | {data.num_edges} edges | {data.num_node_features} feats")
    return data

def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i,n): m=torch.zeros(n,dtype=torch.bool); m[i]=True; return m
    return mk(tr,len(y)), mk(va,len(y)), mk(te,len(y))

def temporal_client_split(data, gtm, nc=4, tr=0.3):
    labels, ts = data.y.cpu().numpy(), data.timestep.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(ts[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp]==1], sp[labels[sp]==0]
        def ss(a):
            if len(a)==0: return a,a
            n=max(1,int(len(a)*tr)); return a[:-n],a[-n:]
        it,ie = ss(ill); lt,le = ss(lic)
        t,e = np.concatenate([it,lt]), np.concatenate([ie,le])
        mt,me = torch.zeros(data.num_nodes,dtype=torch.bool), torch.zeros(data.num_nodes,dtype=torch.bool)
        mt[t]=True; me[e]=True
        clients.append({'id':i,'train_mask':mt,'test_mask':me,'n_train':len(t),'n_test':len(e)})
    return clients

def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

def get_inductive_ei(data, tmask, temask, dev):
    ei,tr,te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr|te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

# =============================================================================
# Model
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e*2), nn.ReLU(), nn.Linear(e*2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp,ei)+self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1,ei)+self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x,ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)

class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1,h2 = max(128,in_dim*2), max(64,in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim,h1),nn.BatchNorm1d(h1),nn.ReLU(),nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1,h2),nn.ReLU(),nn.Dropout(cfg.head_dropout/2),nn.Linear(h2,2))
    def forward(self,x): return self.net(x)

class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb

# =============================================================================
# Losses & Federation
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2*counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1-1e-7)
    return (alpha[labels].to(device) * (1-pt)**gamma * ce).mean()

def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y>=0)
    return {c: emb[labeled & (labels==c)].mean(0) if (labeled & (labels==c)).sum()>0
            else torch.zeros(emb.shape[1], device=dev) for c in [0,1]}

def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev)>=0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)

def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    w = 1.0/len(budgets)
    return sum(w * proto_supcon(z[:,:d], labels, mask, {c:v[:d] for c,v in gprotos.items()}, dev, tau) for d in budgets)

def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0,1]}
        else:
            pred = client_ps[i-1]
            bl = {}
            for c in [0,1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c==1 else 0.3
                bl[c] = F.normalize((al*p + (1-al)*o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out

def fedavg(gm, lms, sizes):
    excl = ['running_mean','running_var','num_batches_tracked']
    tot = sum(sizes); ns = {}
    for k,v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k]=v; continue
        acc = torch.zeros_like(v.float())
        for i,lm in enumerate(lms): acc += lm.state_dict()[k].float() * (sizes[i]/tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)

def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0]+list(tdims); seg = {0:[], 1:[]}
    for lo,hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i]>=hi] or [i for i in range(len(cprotos)) if ctiers[i]==max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i]**0.5); continue
            ls,gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm()<1e-8 or gs.norm()<1e-8 else cfg.contrib_floor+(1-cfg.contrib_floor)*(F.cosine_similarity(ls.unsqueeze(0),gs.unsqueeze(0),eps=1e-5).item()+1)/2
            ws.append((sizes[i]**0.5)*q)
        tw = sum(ws)+1e-8
        for c in [0,1]:
            acc = torch.zeros(hi-lo, device=cprotos[0][c].device)
            for i,w in zip(elig,ws): acc += (w/tw)*cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0,1]}

# =============================================================================
# Evaluation
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1':0.,'auc':0.,'prec':0.,'rec':0.,'probs':zeros,'true':zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:,1].cpu().numpy()
    if not np.isfinite(probs_tr).all(): return FAIL

    tr_lab = tmask.cpu() & (y_cpu>=0); best_thresh = 0.5
    if tr_lab.sum()>0 and len(np.unique(y_cpu[tr_lab].numpy()))>1:
        p,r,th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2*p*r/(p+r+1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:,1].cpu().numpy()
    if not np.isfinite(probs_te).all(): return FAIL

    te_lab = temask.cpu() & (y_cpu>=0)
    if te_lab.sum()==0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt,pd_,zero_division=0),
            'auc': roc_auc_score(tt,pm) if len(np.unique(tt))>1 else 0.,
            'prec': precision_score(tt,pd_,zero_division=0),
            'rec': recall_score(tt,pd_,zero_division=0), 'probs':pm, 'true':tt}

def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0,1,n_bins+1), 0.0
    for i in range(n_bins):
        m = (probs>=bins[i]) & (probs<bins[i+1])
        if m.sum()>0: ece += (m.sum()/len(true))*abs(true[m].mean()-probs[m].mean())
    return float(ece)

def avg_metrics(ml):
    out = {}
    for k in ['f1','auc','prec','rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k+'_std'] = float(np.mean(vs)), float(np.std(vs))
    return out

# =============================================================================
# Shared AMP training step
# =============================================================================
def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()

# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler()
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y>=0)
        lb = data.y[vm].to(dev)

        best_f1, patience = 0, 0
        for ep in range(600):
            model.train(); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            if (ep+1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg

# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    best_f1, best_state, patience = 0, gm.state_dict(), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        # Compute prototypes in float32
        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler()
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                # Supervised loss in AMP
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                # Prototype loss in FLOAT32 (outside autocast to prevent NaN)
                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models  # free GPU memory

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        if m['f1'] > best_f1: best_f1 = m['f1']; best_state = gm.state_dict(); patience = 0
        else: patience += 1
        if (rnd+1) % 5 == 0:
            print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt

# =============================================================================
# Method 3: NEST-64→8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    prev_protos, best_f1, best_state, patience, total_comm = None, 0, gm.state_dict(), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c,v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler()
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n:p.detach().clone() for n,p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                # Supervised + FedProx in AMP
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p-enc_ref[n])**2).sum() for n,p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder/2)*prox

                # Multi-budget prototype loss in FLOAT32 (outside autocast)
                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d*2*4*2 for d in ctiers)

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        if m['f1'] > best_f1: best_f1=m['f1']; best_state=gm.state_dict(); patience=0
        else: patience += 1
        if (rnd+1) % 5 == 0:
            print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)

    # FedPer head finetune
    for p in gm.encoder.parameters(): p.requires_grad_(False)
    for ft_rnd in range(cfg.head_finetune_rounds):
        for c in clients:
            opt = AdamW(gm.head.parameters(), lr=cfg.lr_min*5)
            scaler = torch.cuda.amp.GradScaler()
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            for _ in range(5):
                gm.train(); opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    lo, _ = gm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(lo[vm], lb, dev)
                amp_step(gm, opt, scaler, loss)
    for p in gm.encoder.parameters(): p.requires_grad_(True)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'
    data = load_and_build_graph(DATA_DIR).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = temporal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} IEEE-CIS External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-"*95)
    for method, runs in results.items():
        f1m,f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am,as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm,rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs])/1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")
    print(f"\nPrior 3-fold CV sanity check: F1 = 0.6232 ± 0.0070, AUC = 0.8991 ± 0.0002")

In [3]:

import os, glob, time, random, copy, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()


def autocast_ctx():
    """AMP only makes sense (and only works) on CUDA. On CPU this is a no-op
    context manager so the exact same training code runs on either device --
    the Ethereum dataset is tiny (~10K rows), so CPU-only Kaggle sessions
    should be able to run this fine without a GPU at all."""
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return contextlib.nullcontext()


# =============================================================================
# Configuration (identical to the IEEE-CIS run for cross-dataset comparability)
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    global_rounds: int = 100;  head_finetune_rounds: int = 20
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 25
    lr: float = 0.005;  lr_min: float = 0.0005
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    lam_max: float = 0.6;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)


# =============================================================================
# Data -- Ethereum Fraud Detection (Kaggle: vagifa/ethereum-frauddetection-dataset)
# =============================================================================
def load_and_build_graph_eth(csv_path, k_neighbors=8):
    """
    CAVEAT (read before trusting these numbers in the paper): unlike IEEE-CIS
    (card1 groups + real timestamps) or Elliptic (a real transaction graph),
    this dataset ships as a flat table of per-address on-chain activity
    features with NO edge list and NO timestamp column. Two substitutions are
    made here, both worth stating explicitly in the writeup:

    1. GRAPH CONSTRUCTION: since there's no natural linking key between
       addresses, edges are built as a k-nearest-neighbor graph in scaled
       feature space. This is standard practice for applying GNNs to tabular
       fraud data (fraudulent addresses tend to cluster in feature space --
       similar transaction volume/frequency/counterparty patterns), but it is
       a PROXY graph, not an observed relational structure. Report it as such.
    2. NON-IID CLIENT SPLIT: with no timestamp, clients are formed by sorting
       addresses on 'Time Diff between first and last (Mins)' (an on-chain
       activity-recency proxy) instead of chronological order, mirroring the
       temporal-split framing used for IEEE-CIS/Elliptic on a different axis.
    """
    print("Loading Ethereum Fraud Detection data...")
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()  # this dataset ships with stray leading/trailing spaces in headers

    for drop_col in ['Unnamed: 0', 'Index', 'Address']:
        if drop_col in df.columns:
            df = df.drop(columns=[drop_col])

    assert 'FLAG' in df.columns, "Expected a 'FLAG' target column -- check the CSV headers after stripping."
    y = df['FLAG'].values.astype(np.int64)

    # The two ERC20-token-type columns are genuinely categorical (token symbol
    # strings); everything else is numeric (coerced defensively in case any
    # column has stray blank/non-numeric entries).
    cat_cols = [c for c in ['ERC20 most sent token type', 'ERC20_most_rec_token_type'] if c in df.columns]
    drop_cols = ['FLAG']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols]

    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    # Missing-value handling: indicator + median impute (same pattern used for
    # IEEE-CIS -- do NOT feed a sentinel like -999 into StandardScaler).
    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'
            df[ic] = df[col].isna().astype(np.float32)
            miss_cols.append(ic)
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val if pd.notna(median_val) else 0.0)

    # This dataset also has some columns that are all-zero/constant for most
    # rows (e.g. contract-related fields for addresses that never touched a
    # contract) -- StandardScaler handles that fine (std floor), no special
    # handling needed beyond the NaN guard below.

    order_col_candidates = ['Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx']
    order_col = next((c for c in order_col_candidates if c in num_cols), num_cols[0])

    feat_cols = num_cols + cat_cols + miss_cols
    X_raw = df[feat_cols].values.astype(np.float32)
    X = np.nan_to_num(StandardScaler().fit_transform(X_raw))
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"  {len(df)} addresses | {X.shape[1]} features | fraud rate = {y.mean():.4f}")

    # --- k-NN graph construction (proxy relational structure, see CAVEAT) ---
    print(f"  Building k-NN graph (k={k_neighbors}) in scaled feature space...")
    nbrs = NearestNeighbors(n_neighbors=k_neighbors + 1, algorithm='auto').fit(X)
    dists, idxs = nbrs.kneighbors(X)
    src, dst, edge_d = [], [], []
    for i in range(X.shape[0]):
        for j in range(1, k_neighbors + 1):  # skip self (column 0)
            nb = idxs[i, j]
            d = dists[i, j]
            src.append(i); dst.append(nb); edge_d.append(d)
            src.append(nb); dst.append(i); edge_d.append(d)

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr = torch.tensor(np.log1p(np.array(edge_d, dtype=np.float32))).unsqueeze(-1)

    order_key = torch.tensor(np.nan_to_num(df[order_col].values.astype(np.float32)), dtype=torch.float)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=edge_index,
                edge_attr=edge_attr,
                y=torch.tensor(y, dtype=torch.long),
                order_key=order_key)
    print(f"  Graph: {data.num_nodes} nodes | {data.num_edges} edges | "
          f"{data.num_node_features} feats | order_col='{order_col}'")
    return data


def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i, n): m = torch.zeros(n, dtype=torch.bool); m[i] = True; return m
    return mk(tr, len(y)), mk(va, len(y)), mk(te, len(y))


def ordinal_client_split(data, gtm, nc=4, tr=0.3):
    """Non-IID client split for datasets without a real timestamp: sorts by
    data.order_key instead of chronological time (see CAVEAT in
    load_and_build_graph_eth). Structurally identical to the
    temporal_client_split used for IEEE-CIS/Elliptic, just on a different axis."""
    labels, order = data.y.cpu().numpy(), data.order_key.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(order[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp] == 1], sp[labels[sp] == 0]
        def ss(a):
            if len(a) == 0: return a, a
            n = max(1, int(len(a) * tr)); return a[:-n], a[-n:]
        it, ie = ss(ill); lt, le = ss(lic)
        t, e = np.concatenate([it, lt]), np.concatenate([ie, le])
        mt, me = torch.zeros(data.num_nodes, dtype=torch.bool), torch.zeros(data.num_nodes, dtype=torch.bool)
        mt[t] = True; me[e] = True
        clients.append({'id': i, 'train_mask': mt, 'test_mask': me, 'n_train': len(t), 'n_test': len(e)})
        print(f"    Client {i}: train={len(t):4d} test={len(e):3d} "
              f"fraud_train={int(labels[t].sum())} fraud_test={int(labels[e].sum())}")
    return clients


def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:, k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)


def get_inductive_ei(data, tmask, temask, dev):
    ei, tr, te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr | te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:, k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)


# =============================================================================
# Model (unchanged from the IEEE-CIS run)
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp, ei) + self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1, ei) + self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x, ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)


class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1, h2 = max(128, in_dim * 2), max(64, in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(cfg.head_dropout / 2), nn.Linear(h2, 2))
    def forward(self, x): return self.net(x)


class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb


# =============================================================================
# Losses & Federation (unchanged from the IEEE-CIS run)
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2 * counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1 - 1e-7)
    return (alpha[labels].to(device) * (1 - pt) ** gamma * ce).mean()


def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y >= 0)
    return {c: emb[labeled & (labels == c)].mean(0) if (labeled & (labels == c)).sum() > 0
            else torch.zeros(emb.shape[1], device=dev) for c in [0, 1]}


def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev) >= 0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)


def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    w = 1.0 / len(budgets)
    return sum(w * proto_supcon(z[:, :d], labels, mask, {c: v[:d] for c, v in gprotos.items()}, dev, tau) for d in budgets)


def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0, 1]}
        else:
            pred = client_ps[i - 1]
            bl = {}
            for c in [0, 1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c == 1 else 0.3
                bl[c] = F.normalize((al * p + (1 - al) * o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out


def fedavg(gm, lms, sizes):
    excl = ['running_mean', 'running_var', 'num_batches_tracked']
    tot = sum(sizes); ns = {}
    for k, v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k] = v; continue
        acc = torch.zeros_like(v.float())
        for i, lm in enumerate(lms): acc += lm.state_dict()[k].float() * (sizes[i] / tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)


def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0] + list(tdims); seg = {0: [], 1: []}
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i] >= hi] or [i for i in range(len(cprotos)) if ctiers[i] == max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i] ** 0.5); continue
            ls, gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm() < 1e-8 or gs.norm() < 1e-8 else cfg.contrib_floor + (1 - cfg.contrib_floor) * (F.cosine_similarity(ls.unsqueeze(0), gs.unsqueeze(0), eps=1e-5).item() + 1) / 2
            ws.append((sizes[i] ** 0.5) * q)
        tw = sum(ws) + 1e-8
        for c in [0, 1]:
            acc = torch.zeros(hi - lo, device=cprotos[0][c].device)
            for i, w in zip(elig, ws): acc += (w / tw) * cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0, 1]}


# =============================================================================
# Evaluation (unchanged from the IEEE-CIS run)
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0., 'probs': zeros, 'true': zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:, 1].float().cpu().numpy()
    if not np.isfinite(probs_tr).all(): return FAIL

    tr_lab = tmask.cpu() & (y_cpu >= 0); best_thresh = 0.5
    if tr_lab.sum() > 0 and len(np.unique(y_cpu[tr_lab].numpy())) > 1:
        p, r, th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2 * p * r / (p + r + 1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:, 1].float().cpu().numpy()
    if not np.isfinite(probs_te).all(): return FAIL

    te_lab = temask.cpu() & (y_cpu >= 0)
    if te_lab.sum() == 0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt, pd_, zero_division=0),
            'auc': roc_auc_score(tt, pm) if len(np.unique(tt)) > 1 else 0.,
            'prec': precision_score(tt, pd_, zero_division=0),
            'rec': recall_score(tt, pd_, zero_division=0), 'probs': pm, 'true': tt}


def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for i in range(n_bins):
        m = (probs >= bins[i]) & (probs < bins[i + 1])
        if m.sum() > 0: ece += (m.sum() / len(true)) * abs(true[m].mean() - probs[m].mean())
    return float(ece)


def avg_metrics(ml):
    out = {}
    for k in ['f1', 'auc', 'prec', 'rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k + '_std'] = float(np.mean(vs)), float(np.std(vs))
    return out


def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()


# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y >= 0)
        lb = data.y[vm].to(dev)

        best_f1, patience = 0, 0
        for ep in range(600):
            model.train(); opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            if (ep + 1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg


# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    best_f1, best_state, patience = 0, gm.state_dict(), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y >= 0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        if m['f1'] > best_f1: best_f1 = m['f1']; best_state = gm.state_dict(); patience = 0
        else: patience += 1
        if (rnd + 1) % 5 == 0:
            print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt


# =============================================================================
# Method 3: NEST-64->8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    prev_protos, best_f1, best_state, patience, total_comm = None, 0, gm.state_dict(), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c, v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y >= 0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n: p.detach().clone() for n, p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p - enc_ref[n]) ** 2).sum() for n, p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder / 2) * prox

                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d * 2 * 4 * 2 for d in ctiers)

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        if m['f1'] > best_f1: best_f1 = m['f1']; best_state = gm.state_dict(); patience = 0
        else: patience += 1
        if (rnd + 1) % 5 == 0:
            print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)

    for p in gm.encoder.parameters(): p.requires_grad_(False)
    for ft_rnd in range(cfg.head_finetune_rounds):
        for c in clients:
            opt = AdamW(gm.head.parameters(), lr=cfg.lr_min * 5)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y >= 0)
            lb = data.y[vm].to(dev)
            for _ in range(5):
                gm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    lo, _ = gm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(lo[vm], lb, dev)
                amp_step(gm, opt, scaler, loss)
    for p in gm.encoder.parameters(): p.requires_grad_(True)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt


# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    # Update this path to wherever the Ethereum Fraud Detection CSV is mounted
    # in your Kaggle input (e.g. the vagifa/ethereum-frauddetection-dataset
    # dataset's transaction_dataset.csv).
    CSV_PATH = '/kaggle/input/datasets/vagifa/ethereum-frauddetection-dataset/transaction_dataset.csv'

    data = load_and_build_graph_eth(CSV_PATH, k_neighbors=8).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = ordinal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} Ethereum Fraud Detection External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-" * 95)
    for method, runs in results.items():
        f1m, f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am, as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm, rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs]) / 1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")

Loading Ethereum Fraud Detection data...
  9841 addresses | 70 features | fraud rate = 0.2214
  Building k-NN graph (k=8) in scaled feature space...
  Graph: 9841 nodes | 157456 edges | 70 feats | order_col='Time Diff between first and last (Mins)'
    Client 0: train=1034 test=442 fraud_train=284 fraud_test=121
    Client 1: train=1034 test=442 fraud_train=401 fraud_test=171
    Client 2: train=1034 test=442 fraud_train=192 fraud_test=82
    Client 3: train=1034 test=442 fraud_train=40 fraud_test=16

========================= SEED 42 =========================
  [LocalOnly]
    Client 0: F1=0.7768 AUC=0.9340
    Client 1: F1=0.6233 AUC=0.9071
    Client 2: F1=0.8928 AUC=0.9854
    Client 3: F1=0.7697 AUC=0.9755
  => F1: 0.7657, AUC: 0.9505
  [FedProto-8]
    Round   5: F1=0.6092 AUC=0.8813 (best=0.7563, pat=1)
    Round  10: F1=0.5076 AUC=0.6891 (best=0.8266, pat=1)
    Round  15: F1=0.0000 AUC=0.0000 (best=0.8266, pat=6)
    Round  20: F1=0.0000 AUC=0.0000 (best=0.8266, pat=11)
    Ea

In [ ]:
!pip install torch_geometric

In [ ]:
import subprocess, sys, importlib

def get_torch_cuda_tag():
    import torch
    tv = torch.__version__.split('+')[0]
    cv = torch.version.cuda
    if cv is None:
        return tv, 'cpu'
    major, minor = cv.split('.')[:2]
    return tv, f'cu{major}{minor}'

torch_ver, cuda_tag = get_torch_cuda_tag()
WHL_URL = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'



# sanity check BEFORE running anything else
import torch_geometric
from torch_geometric.nn import GATv2Conv, SAGEConv
print('torch_geometric', torch_geometric.__version__, 'OK')

In [1]:
import torch_geometric
from torch_geometric.nn import GATv2Conv, SAGEConv
print('torch_geometric', torch_geometric.__version__, 'OK — no torch_cluster/scatter/sparse needed')

torch_geometric 2.5.3 OK — no torch_cluster/scatter/sparse needed


In [4]:
import os, glob, time, random, copy, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

def autocast_ctx():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return contextlib.nullcontext()

# =============================================================================
# Configuration 
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    global_rounds: int = 100;  head_finetune_rounds: int = 20
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 25
    lr: float = 0.005;  lr_min: float = 0.0005
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    lam_max: float = 0.6;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# =============================================================================
# Data
# =============================================================================
def load_and_build_graph_eth(csv_path, k_neighbors=8):
    print("Loading Ethereum Fraud Detection data...")
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip() 

    for drop_col in ['Unnamed: 0', 'Index', 'Address']:
        if drop_col in df.columns:
            df = df.drop(columns=[drop_col])

    assert 'FLAG' in df.columns, "Expected a 'FLAG' target column"
    y = df['FLAG'].values.astype(np.int64)

    cat_cols = [c for c in ['ERC20 most sent token type', 'ERC20_most_rec_token_type'] if c in df.columns]
    drop_cols = ['FLAG']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols]

    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'
            df[ic] = df[col].isna().astype(np.float32)
            miss_cols.append(ic)
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val if pd.notna(median_val) else 0.0)

    order_col_candidates = ['Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx']
    order_col = next((c for c in order_col_candidates if c in num_cols), num_cols[0])

    feat_cols = num_cols + cat_cols + miss_cols
    X_raw = df[feat_cols].values.astype(np.float32)
    X = np.nan_to_num(StandardScaler().fit_transform(X_raw))
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"  {len(df)} addresses | {X.shape[1]} features | fraud rate = {y.mean():.4f}")

    print(f"  Building k-NN graph (k={k_neighbors}) in scaled feature space...")
    nbrs = NearestNeighbors(n_neighbors=k_neighbors + 1, algorithm='auto').fit(X)
    dists, idxs = nbrs.kneighbors(X)
    src, dst, edge_d = [], [], []
    for i in range(X.shape[0]):
        for j in range(1, k_neighbors + 1):
            nb = idxs[i, j]
            d = dists[i, j]
            src.append(i); dst.append(nb); edge_d.append(d)
            src.append(nb); dst.append(i); edge_d.append(d)

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr = torch.tensor(np.log1p(np.array(edge_d, dtype=np.float32))).unsqueeze(-1)
    order_key = torch.tensor(np.nan_to_num(df[order_col].values.astype(np.float32)), dtype=torch.float)

    data = Data(x=torch.tensor(X, dtype=torch.float), edge_index=edge_index,
                edge_attr=edge_attr, y=torch.tensor(y, dtype=torch.long), order_key=order_key)
    
    print(f"  Graph: {data.num_nodes} nodes | {data.num_edges} edges | "
          f"{data.num_node_features} feats | order_col='{order_col}'")
    return data

def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i, n): m = torch.zeros(n, dtype=torch.bool); m[i] = True; return m
    return mk(tr, len(y)), mk(va, len(y)), mk(te, len(y))

def ordinal_client_split(data, gtm, nc=4, tr=0.3):
    labels, order = data.y.cpu().numpy(), data.order_key.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(order[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp] == 1], sp[labels[sp] == 0]
        def ss(a):
            if len(a) == 0: return a, a
            n = max(1, int(len(a) * tr)); return a[:-n], a[-n:]
        it, ie = ss(ill); lt, le = ss(lic)
        t, e = np.concatenate([it, lt]), np.concatenate([ie, le])
        mt, me = torch.zeros(data.num_nodes, dtype=torch.bool), torch.zeros(data.num_nodes, dtype=torch.bool)
        mt[t] = True; me[e] = True
        clients.append({'id': i, 'train_mask': mt, 'test_mask': me, 'n_train': len(t), 'n_test': len(e)})
        print(f"    Client {i}: train={len(t):4d} test={len(e):3d} "
              f"fraud_train={int(labels[t].sum())} fraud_test={int(labels[e].sum())}")
    return clients

def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:, k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

def get_inductive_ei(data, tmask, temask, dev):
    ei, tr, te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr | te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:, k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

# =============================================================================
# Model
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e * 2), nn.ReLU(), nn.Linear(e * 2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp, ei) + self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1, ei) + self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x, ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)

class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1, h2 = max(128, in_dim * 2), max(64, in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(cfg.head_dropout / 2), nn.Linear(h2, 2))
    def forward(self, x): return self.net(x)

class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb

# =============================================================================
# Losses & Federation
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2 * counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1 - 1e-7)
    return (alpha[labels].to(device) * (1 - pt) ** gamma * ce).mean()

def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y >= 0)
    return {c: emb[labeled & (labels == c)].mean(0) if (labeled & (labels == c)).sum() > 0
            else torch.zeros(emb.shape[1], device=dev) for c in [0, 1]}

def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev) >= 0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)

def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    w = 1.0 / len(budgets)
    return sum(w * proto_supcon(z[:, :d], labels, mask, {c: v[:d] for c, v in gprotos.items()}, dev, tau) for d in budgets)

def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0, 1]}
        else:
            pred = client_ps[i - 1]
            bl = {}
            for c in [0, 1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c == 1 else 0.3
                bl[c] = F.normalize((al * p + (1 - al) * o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out

def fedavg(gm, lms, sizes):
    excl = ['running_mean', 'running_var', 'num_batches_tracked']
    
    valid_lms, valid_sizes = [], []
    for i, lm in enumerate(lms):
        if all(torch.isfinite(p).all() for p in lm.parameters()):
            valid_lms.append(lm)
            valid_sizes.append(sizes[i])
        else:
            print(f"      [WARNING] Client {i} diverged to NaNs! Dropping from aggregation.")
            
    if not valid_lms:
        print("      [FATAL] All clients diverged to NaNs this round!")
        return

    tot = sum(valid_sizes); ns = {}
    for k, v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k] = v; continue
        acc = torch.zeros_like(v.float())
        for i, lm in enumerate(valid_lms): 
            acc += lm.state_dict()[k].float() * (valid_sizes[i] / tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)

def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0] + list(tdims); seg = {0: [], 1: []}
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i] >= hi] or [i for i in range(len(cprotos)) if ctiers[i] == max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i] ** 0.5); continue
            ls, gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm() < 1e-8 or gs.norm() < 1e-8 else cfg.contrib_floor + (1 - cfg.contrib_floor) * (F.cosine_similarity(ls.unsqueeze(0), gs.unsqueeze(0), eps=1e-5).item() + 1) / 2
            ws.append((sizes[i] ** 0.5) * q)
        tw = sum(ws) + 1e-8
        for c in [0, 1]:
            acc = torch.zeros(hi - lo, device=cprotos[0][c].device)
            for i, w in zip(elig, ws): acc += (w / tw) * cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0, 1]}

# =============================================================================
# Evaluation
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1': 0., 'auc': 0., 'prec': 0., 'rec': 0., 'probs': zeros, 'true': zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:, 1].float().cpu().numpy()
    
    if not np.isfinite(probs_tr).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs!")
        return FAIL

    tr_lab = tmask.cpu() & (y_cpu >= 0); best_thresh = 0.5
    if tr_lab.sum() > 0 and len(np.unique(y_cpu[tr_lab].numpy())) > 1:
        p, r, th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2 * p * r / (p + r + 1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:, 1].float().cpu().numpy()
    
    if not np.isfinite(probs_te).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs on test set!")
        return FAIL

    te_lab = temask.cpu() & (y_cpu >= 0)
    if te_lab.sum() == 0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt, pd_, zero_division=0),
            'auc': roc_auc_score(tt, pm) if len(np.unique(tt)) > 1 else 0.,
            'prec': precision_score(tt, pd_, zero_division=0),
            'rec': recall_score(tt, pd_, zero_division=0), 'probs': pm, 'true': tt}

def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for i in range(n_bins):
        m = (probs >= bins[i]) & (probs < bins[i + 1])
        if m.sum() > 0: ece += (m.sum() / len(true)) * abs(true[m].mean() - probs[m].mean())
    return float(ece)

def avg_metrics(ml):
    out = {}
    for k in ['f1', 'auc', 'prec', 'rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k + '_std'] = float(np.mean(vs)), float(np.std(vs))
    return out

def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()

# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y >= 0)
        lb = data.y[vm].to(dev)

        best_f1, patience = 0, 0
        for ep in range(600):
            model.train(); opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            if (ep + 1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg

# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    best_f1, best_state, patience = 0, copy.deepcopy(gm.state_dict()), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y >= 0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        
        if m['f1'] > best_f1: 
            best_f1 = m['f1']
            best_state = copy.deepcopy(gm.state_dict())
            patience = 0
            print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        else: 
            patience += 1
            if (rnd + 1) % 5 == 0:
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt

# =============================================================================
# Method 3: NEST-64->8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    prev_protos, best_f1, best_state, patience, total_comm = None, 0, copy.deepcopy(gm.state_dict()), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c, v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y >= 0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n: p.detach().clone() for n, p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p - enc_ref[n]) ** 2).sum() for n, p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder / 2) * prox

                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d * 2 * 4 * 2 for d in ctiers)

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        
        if m['f1'] > best_f1: 
            best_f1 = m['f1']
            best_state = copy.deepcopy(gm.state_dict())
            patience = 0
            print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        else: 
            patience += 1
            if (rnd + 1) % 5 == 0:
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)

    for p in gm.encoder.parameters(): p.requires_grad_(False)
    for ft_rnd in range(cfg.head_finetune_rounds):
        for c in clients:
            opt = AdamW(gm.head.parameters(), lr=cfg.lr_min * 5)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y >= 0)
            lb = data.y[vm].to(dev)
            for _ in range(5):
                gm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    lo, _ = gm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(lo[vm], lb, dev)
                amp_step(gm, opt, scaler, loss)
    for p in gm.encoder.parameters(): p.requires_grad_(True)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    # Update this path to wherever the Ethereum Fraud Detection CSV is mounted
    # in your Kaggle input
    CSV_PATH = '/kaggle/input/datasets/vagifa/ethereum-frauddetection-dataset/transaction_dataset.csv'

    data = load_and_build_graph_eth(CSV_PATH, k_neighbors=8).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = ordinal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} Ethereum Fraud Detection External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-" * 95)
    for method, runs in results.items():
        f1m, f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am, as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm, rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs]) / 1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")

Loading Ethereum Fraud Detection data...
  9841 addresses | 70 features | fraud rate = 0.2214
  Building k-NN graph (k=8) in scaled feature space...
  Graph: 9841 nodes | 157456 edges | 70 feats | order_col='Time Diff between first and last (Mins)'
    Client 0: train=1034 test=442 fraud_train=284 fraud_test=121
    Client 1: train=1034 test=442 fraud_train=401 fraud_test=171
    Client 2: train=1034 test=442 fraud_train=192 fraud_test=82
    Client 3: train=1034 test=442 fraud_train=40 fraud_test=16

========================= SEED 42 =========================
  [LocalOnly]
    Client 0: F1=0.7745 AUC=0.9262
    Client 1: F1=0.6290 AUC=0.9305
    Client 2: F1=0.9141 AUC=0.9904
    Client 3: F1=0.8836 AUC=0.9823
  => F1: 0.8003, AUC: 0.9574
  [FedProto-8]
    [NEW BEST] Round   1: F1=0.5663 AUC=0.9737
    [NEW BEST] Round   2: F1=0.6177 AUC=0.9485
    Round   5: F1=0.5064 AUC=0.8176 (best=0.6177, pat=3)
    [NEW BEST] Round   7: F1=0.6658 AUC=0.8876
    Round  10: F1=0.4773 AUC=0.6717 (

In [6]:
import os, glob, time, random, copy, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

def autocast_ctx():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return contextlib.nullcontext()

# =============================================================================
# Configuration
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    global_rounds: int = 100;  head_finetune_rounds: int = 20
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 25
    lr: float = 0.005;  lr_min: float = 0.0005
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    lam_max: float = 0.6;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# =============================================================================
# Data
# =============================================================================
def load_and_build_graph(data_dir, card_time_window=1):
    print("Loading IEEE-CIS Data...")
    txn = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    idn = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))
    df = pd.merge(txn, idn, on='TransactionID', how='left')
    y = df['isFraud'].values

    cat_cols = list(set(list(df.select_dtypes(include=['object']).columns) +
                        [c for c in ['card1','card2','card3','card4','card5','card6',
                                     'addr1','addr2','P_emaildomain','R_emaildomain'] if c in df.columns]))
    drop_cols = ['isFraud', 'TransactionID']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'; df[ic] = df[col].isna().astype(np.float32); miss_cols.append(ic)
            df[col] = df[col].fillna(df[col].median())

    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    agg_cols = []
    for ac in ['TransactionAmt', 'TransactionDT']:
        if ac in df.columns:
            grp = df.groupby('card1')[ac]
            mc, sc, cc = f'{ac}_c1_mean', f'{ac}_c1_std', f'{ac}_c1_count'
            df[mc] = grp.transform('mean'); df[sc] = grp.transform('std').fillna(0); df[cc] = grp.transform('count')
            agg_cols.extend([mc, sc, cc])
    if 'TransactionAmt' in df.columns and 'TransactionAmt_c1_mean' in df.columns:
        df['Amt_c1_dev'] = df['TransactionAmt'] - df['TransactionAmt_c1_mean']; agg_cols.append('Amt_c1_dev')

    feat_cols = [c for c in num_cols if c != 'TransactionDT'] + cat_cols + miss_cols + agg_cols
    X = np.nan_to_num(StandardScaler().fit_transform(df[feat_cols].values.astype(np.float32)))

    df['orig_idx'] = np.arange(len(df))
    src, dst, edge_dt = [], [], []
    for _, g in df.sort_values(['card1','TransactionDT']).groupby('card1'):
        idxs, times = g['orig_idx'].values, g['TransactionDT'].values
        if len(idxs) < 2: continue
        for off in range(1, min(card_time_window, len(idxs)-1)+1):
            a, b, dt = idxs[:-off], idxs[off:], np.abs(times[off:]-times[:-off]).astype(np.float32)
            src.extend(a); dst.extend(b); edge_dt.extend(dt)
            src.extend(b); dst.extend(a); edge_dt.extend(dt)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=torch.tensor([src,dst], dtype=torch.long),
                edge_attr=torch.tensor(np.log1p(np.array(edge_dt, dtype=np.float32))).unsqueeze(-1),
                y=torch.tensor(y, dtype=torch.long),
                timestep=torch.tensor(df['TransactionDT'].values, dtype=torch.long))
    print(f"Graph: {data.num_nodes} nodes | {data.num_edges} edges | {data.num_node_features} feats")
    return data

def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i,n): m=torch.zeros(n,dtype=torch.bool); m[i]=True; return m
    return mk(tr,len(y)), mk(va,len(y)), mk(te,len(y))

def temporal_client_split(data, gtm, nc=4, tr=0.3):
    labels, ts = data.y.cpu().numpy(), data.timestep.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(ts[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp]==1], sp[labels[sp]==0]
        def ss(a):
            if len(a)==0: return a,a
            n=max(1,int(len(a)*tr)); return a[:-n],a[-n:]
        it,ie = ss(ill); lt,le = ss(lic)
        t,e = np.concatenate([it,lt]), np.concatenate([ie,le])
        mt,me = torch.zeros(data.num_nodes,dtype=torch.bool), torch.zeros(data.num_nodes,dtype=torch.bool)
        mt[t]=True; me[e]=True
        clients.append({'id':i,'train_mask':mt,'test_mask':me,'n_train':len(t),'n_test':len(e)})
    return clients

def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

def get_inductive_ei(data, tmask, temask, dev):
    ei,tr,te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr|te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

# =============================================================================
# Model
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e*2), nn.ReLU(), nn.Linear(e*2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp,ei)+self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1,ei)+self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x,ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)

class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1,h2 = max(128,in_dim*2), max(64,in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim,h1),nn.BatchNorm1d(h1),nn.ReLU(),nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1,h2),nn.ReLU(),nn.Dropout(cfg.head_dropout/2),nn.Linear(h2,2))
    def forward(self,x): return self.net(x)

class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb

# =============================================================================
# Losses & Federation
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2*counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1-1e-7)
    return (alpha[labels].to(device) * (1-pt)**gamma * ce).mean()

def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y>=0)
    return {c: emb[labeled & (labels==c)].mean(0) if (labeled & (labels==c)).sum()>0
            else torch.zeros(emb.shape[1], device=dev) for c in [0,1]}

def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev)>=0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)

def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    w = 1.0/len(budgets)
    return sum(w * proto_supcon(z[:,:d], labels, mask, {c:v[:d] for c,v in gprotos.items()}, dev, tau) for d in budgets)

def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0,1]}
        else:
            pred = client_ps[i-1]
            bl = {}
            for c in [0,1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c==1 else 0.3
                bl[c] = F.normalize((al*p + (1-al)*o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out

def fedavg(gm, lms, sizes):
    excl = ['running_mean','running_var','num_batches_tracked']
    
    valid_lms, valid_sizes = [], []
    for i, lm in enumerate(lms):
        if all(torch.isfinite(p).all() for p in lm.parameters()):
            valid_lms.append(lm)
            valid_sizes.append(sizes[i])
        else:
            print(f"      [WARNING] Client {i} diverged to NaNs! Dropping from aggregation.")
            
    if not valid_lms:
        print("      [FATAL] All clients diverged to NaNs this round!")
        return

    tot = sum(valid_sizes); ns = {}
    for k,v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k]=v; continue
        acc = torch.zeros_like(v.float())
        for i,lm in enumerate(valid_lms): 
            acc += lm.state_dict()[k].float() * (valid_sizes[i]/tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)

def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0]+list(tdims); seg = {0:[], 1:[]}
    for lo,hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i]>=hi] or [i for i in range(len(cprotos)) if ctiers[i]==max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i]**0.5); continue
            ls,gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm()<1e-8 or gs.norm()<1e-8 else cfg.contrib_floor+(1-cfg.contrib_floor)*(F.cosine_similarity(ls.unsqueeze(0),gs.unsqueeze(0),eps=1e-5).item()+1)/2
            ws.append((sizes[i]**0.5)*q)
        tw = sum(ws)+1e-8
        for c in [0,1]:
            acc = torch.zeros(hi-lo, device=cprotos[0][c].device)
            for i,w in zip(elig,ws): acc += (w/tw)*cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0,1]}

# =============================================================================
# Evaluation
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1':0.,'auc':0.,'prec':0.,'rec':0.,'probs':zeros,'true':zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_tr).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs!")
        return FAIL

    tr_lab = tmask.cpu() & (y_cpu>=0); best_thresh = 0.5
    if tr_lab.sum()>0 and len(np.unique(y_cpu[tr_lab].numpy()))>1:
        p,r,th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2*p*r/(p+r+1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_te).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs on test set!")
        return FAIL

    te_lab = temask.cpu() & (y_cpu>=0)
    if te_lab.sum()==0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt,pd_,zero_division=0),
            'auc': roc_auc_score(tt,pm) if len(np.unique(tt))>1 else 0.,
            'prec': precision_score(tt,pd_,zero_division=0),
            'rec': recall_score(tt,pd_,zero_division=0), 'probs':pm, 'true':tt}

def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0,1,n_bins+1), 0.0
    for i in range(n_bins):
        m = (probs>=bins[i]) & (probs<bins[i+1])
        if m.sum()>0: ece += (m.sum()/len(true))*abs(true[m].mean()-probs[m].mean())
    return float(ece)

def avg_metrics(ml):
    out = {}
    for k in ['f1','auc','prec','rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k+'_std'] = float(np.mean(vs)), float(np.std(vs))
    return out

# =============================================================================
# Shared AMP training step
# =============================================================================
def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()

# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y>=0)
        lb = data.y[vm].to(dev)

        best_f1, patience = 0, 0
        for ep in range(600):
            model.train(); opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            if (ep+1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg

# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    best_f1, best_state, patience = 0, copy.deepcopy(gm.state_dict()), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        # Compute prototypes
        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models 

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        
        if m['f1'] > best_f1: 
            best_f1 = m['f1']
            best_state = copy.deepcopy(gm.state_dict())
            patience = 0
            print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        else: 
            patience += 1
            if (rnd+1) % 5 == 0:
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt

# =============================================================================
# Method 3: NEST-64→8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    prev_protos, best_f1, best_state, patience, total_comm = None, 0, copy.deepcopy(gm.state_dict()), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c,v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n:p.detach().clone() for n,p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p-enc_ref[n])**2).sum() for n,p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder/2)*prox

                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d*2*4*2 for d in ctiers)

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        
        if m['f1'] > best_f1: 
            best_f1 = m['f1']
            best_state = copy.deepcopy(gm.state_dict())
            patience = 0
            print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        else: 
            patience += 1
            if (rnd+1) % 5 == 0:
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)

    # FedPer head finetune
    for p in gm.encoder.parameters(): p.requires_grad_(False)
    for ft_rnd in range(cfg.head_finetune_rounds):
        for c in clients:
            opt = AdamW(gm.head.parameters(), lr=cfg.lr_min*5)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            for _ in range(5):
                gm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    lo, _ = gm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(lo[vm], lb, dev)
                amp_step(gm, opt, scaler, loss)
    for p in gm.encoder.parameters(): p.requires_grad_(True)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'
    data = load_and_build_graph(DATA_DIR).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = temporal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} IEEE-CIS External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-"*95)
    for method, runs in results.items():
        f1m,f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am,as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm,rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs])/1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")

Loading IEEE-CIS Data...
Graph: 590540 nodes | 1180046 edges | 817 feats

========================= SEED 42 =========================
  [LocalOnly]
    Client 0: F1=0.3848 AUC=0.7679
    Client 1: F1=0.3241 AUC=0.7929
    Client 2: F1=0.3474 AUC=0.8084
    Client 3: F1=0.3436 AUC=0.7880
  => F1: 0.3500, AUC: 0.7893
  [FedProto-8]
    [NEW BEST] Round   1: F1=0.2511 AUC=0.7614
    [NEW BEST] Round   4: F1=0.2742 AUC=0.7894
    Round   5: F1=0.2387 AUC=0.7802 (best=0.2742, pat=1)
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
    Round  10: F1=0.0000 AUC=0.0000 (best=0.2742, pat=6)
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
      [WARNING] Evaluation aborted: Logits diverged to NaNs!
      [WARNING] Evaluation aborte

KeyboardInterrupt: 

In [ ]:
import os, glob, time, random, copy, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# FIXED FOR NUMERICAL STABILITY: Force FP32
# =============================================================================
USE_AMP = False

def autocast_ctx():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return contextlib.nullcontext()

# =============================================================================
# Configuration
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    global_rounds: int = 100;  head_finetune_rounds: int = 20
    
    # FIXED FOR STABILITY: Lower epochs and LR
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 15
    lr: float = 0.001;  lr_min: float = 0.0001
    
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    lam_max: float = 0.6;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# =============================================================================
# Data
# =============================================================================
def load_and_build_graph(data_dir, card_time_window=1):
    print("Loading IEEE-CIS Data...")
    txn = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    idn = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))
    df = pd.merge(txn, idn, on='TransactionID', how='left')
    y = df['isFraud'].values

    cat_cols = list(set(list(df.select_dtypes(include=['object']).columns) +
                        [c for c in ['card1','card2','card3','card4','card5','card6',
                                     'addr1','addr2','P_emaildomain','R_emaildomain'] if c in df.columns]))
    drop_cols = ['isFraud', 'TransactionID']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'; df[ic] = df[col].isna().astype(np.float32); miss_cols.append(ic)
            df[col] = df[col].fillna(df[col].median())

    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    agg_cols = []
    for ac in ['TransactionAmt', 'TransactionDT']:
        if ac in df.columns:
            grp = df.groupby('card1')[ac]
            mc, sc, cc = f'{ac}_c1_mean', f'{ac}_c1_std', f'{ac}_c1_count'
            df[mc] = grp.transform('mean'); df[sc] = grp.transform('std').fillna(0); df[cc] = grp.transform('count')
            agg_cols.extend([mc, sc, cc])
    if 'TransactionAmt' in df.columns and 'TransactionAmt_c1_mean' in df.columns:
        df['Amt_c1_dev'] = df['TransactionAmt'] - df['TransactionAmt_c1_mean']; agg_cols.append('Amt_c1_dev')

    feat_cols = [c for c in num_cols if c != 'TransactionDT'] + cat_cols + miss_cols + agg_cols
    X = np.nan_to_num(StandardScaler().fit_transform(df[feat_cols].values.astype(np.float32)))

    df['orig_idx'] = np.arange(len(df))
    src, dst, edge_dt = [], [], []
    for _, g in df.sort_values(['card1','TransactionDT']).groupby('card1'):
        idxs, times = g['orig_idx'].values, g['TransactionDT'].values
        if len(idxs) < 2: continue
        for off in range(1, min(card_time_window, len(idxs)-1)+1):
            a, b, dt = idxs[:-off], idxs[off:], np.abs(times[off:]-times[:-off]).astype(np.float32)
            src.extend(a); dst.extend(b); edge_dt.extend(dt)
            src.extend(b); dst.extend(a); edge_dt.extend(dt)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=torch.tensor([src,dst], dtype=torch.long),
                edge_attr=torch.tensor(np.log1p(np.array(edge_dt, dtype=np.float32))).unsqueeze(-1),
                y=torch.tensor(y, dtype=torch.long),
                timestep=torch.tensor(df['TransactionDT'].values, dtype=torch.long))
    print(f"Graph: {data.num_nodes} nodes | {data.num_edges} edges | {data.num_node_features} feats")
    return data

def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i,n): m=torch.zeros(n,dtype=torch.bool); m[i]=True; return m
    return mk(tr,len(y)), mk(va,len(y)), mk(te,len(y))

def temporal_client_split(data, gtm, nc=4, tr=0.3):
    labels, ts = data.y.cpu().numpy(), data.timestep.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(ts[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp]==1], sp[labels[sp]==0]
        def ss(a):
            if len(a)==0: return a,a
            n=max(1,int(len(a)*tr)); return a[:-n],a[-n:]
        it,ie = ss(ill); lt,le = ss(lic)
        t,e = np.concatenate([it,lt]), np.concatenate([ie,le])
        mt,me = torch.zeros(data.num_nodes,dtype=torch.bool), torch.zeros(data.num_nodes,dtype=torch.bool)
        mt[t]=True; me[e]=True
        clients.append({'id':i,'train_mask':mt,'test_mask':me,'n_train':len(t),'n_test':len(e)})
    return clients

def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

def get_inductive_ei(data, tmask, temask, dev):
    ei,tr,te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr|te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

# =============================================================================
# Model
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e*2), nn.ReLU(), nn.Linear(e*2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp,ei)+self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1,ei)+self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x,ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)

class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1,h2 = max(128,in_dim*2), max(64,in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim,h1),nn.BatchNorm1d(h1),nn.ReLU(),nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1,h2),nn.ReLU(),nn.Dropout(cfg.head_dropout/2),nn.Linear(h2,2))
    def forward(self,x): return self.net(x)

class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb

# =============================================================================
# Losses & Federation
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2*counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1-1e-7)
    return (alpha[labels].to(device) * (1-pt)**gamma * ce).mean()

def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y>=0)
    return {c: emb[labeled & (labels==c)].mean(0) if (labeled & (labels==c)).sum()>0
            else torch.zeros(emb.shape[1], device=dev) for c in [0,1]}

def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev)>=0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)

def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    w = 1.0/len(budgets)
    return sum(w * proto_supcon(z[:,:d], labels, mask, {c:v[:d] for c,v in gprotos.items()}, dev, tau) for d in budgets)

def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0,1]}
        else:
            pred = client_ps[i-1]
            bl = {}
            for c in [0,1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c==1 else 0.3
                bl[c] = F.normalize((al*p + (1-al)*o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out

def fedavg(gm, lms, sizes):
    excl = ['running_mean','running_var','num_batches_tracked']
    
    valid_lms, valid_sizes = [], []
    for i, lm in enumerate(lms):
        if all(torch.isfinite(p).all() for p in lm.parameters()):
            valid_lms.append(lm)
            valid_sizes.append(sizes[i])
        else:
            print(f"      [WARNING] Client {i} diverged to NaNs! Dropping from aggregation.")
            
    if not valid_lms:
        print("      [FATAL] All clients diverged to NaNs this round!")
        return

    tot = sum(valid_sizes); ns = {}
    for k,v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k]=v; continue
        acc = torch.zeros_like(v.float())
        for i,lm in enumerate(valid_lms): 
            acc += lm.state_dict()[k].float() * (valid_sizes[i]/tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)

def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0]+list(tdims); seg = {0:[], 1:[]}
    for lo,hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i]>=hi] or [i for i in range(len(cprotos)) if ctiers[i]==max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i]**0.5); continue
            ls,gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm()<1e-8 or gs.norm()<1e-8 else cfg.contrib_floor+(1-cfg.contrib_floor)*(F.cosine_similarity(ls.unsqueeze(0),gs.unsqueeze(0),eps=1e-5).item()+1)/2
            ws.append((sizes[i]**0.5)*q)
        tw = sum(ws)+1e-8
        for c in [0,1]:
            acc = torch.zeros(hi-lo, device=cprotos[0][c].device)
            for i,w in zip(elig,ws): acc += (w/tw)*cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0,1]}

# =============================================================================
# Evaluation
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1':0.,'auc':0.,'prec':0.,'rec':0.,'probs':zeros,'true':zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_tr).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs!")
        return FAIL

    tr_lab = tmask.cpu() & (y_cpu>=0); best_thresh = 0.5
    if tr_lab.sum()>0 and len(np.unique(y_cpu[tr_lab].numpy()))>1:
        p,r,th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2*p*r/(p+r+1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_te).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs on test set!")
        return FAIL

    te_lab = temask.cpu() & (y_cpu>=0)
    if te_lab.sum()==0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt,pd_,zero_division=0),
            'auc': roc_auc_score(tt,pm) if len(np.unique(tt))>1 else 0.,
            'prec': precision_score(tt,pd_,zero_division=0),
            'rec': recall_score(tt,pd_,zero_division=0), 'probs':pm, 'true':tt}

def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0,1,n_bins+1), 0.0
    for i in range(n_bins):
        m = (probs>=bins[i]) & (probs<bins[i+1])
        if m.sum()>0: ece += (m.sum()/len(true))*abs(true[m].mean()-probs[m].mean())
    return float(ece)

def avg_metrics(ml):
    out = {}
    for k in ['f1','auc','prec','rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k+'_std'] = float(np.mean(vs)), float(np.std(vs))
    return out

# =============================================================================
# Shared AMP training step
# =============================================================================
def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()

# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y>=0)
        lb = data.y[vm].to(dev)

        best_f1, patience = -float('inf'), 0
        for ep in range(600):
            model.train(); opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            if (ep+1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg

# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    best_f1, best_state, patience = -float('inf'), copy.deepcopy(gm.state_dict()), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        # Compute prototypes
        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models 

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        
        if m['f1'] > best_f1: 
            best_f1 = m['f1']
            best_state = copy.deepcopy(gm.state_dict())
            patience = 0
            print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        else: 
            patience += 1
            if (rnd+1) % 5 == 0:
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt

# =============================================================================
# Method 3: NEST-64→8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    prev_protos, best_f1, best_state, patience, total_comm = None, -float('inf'), copy.deepcopy(gm.state_dict()), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c,v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n:p.detach().clone() for n,p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p-enc_ref[n])**2).sum() for n,p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder/2)*prox

                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d*2*4*2 for d in ctiers)

        m = evaluate_tuned(gm, data, m_tr, m_val, dev)
        
        if m['f1'] > best_f1: 
            best_f1 = m['f1']
            best_state = copy.deepcopy(gm.state_dict())
            patience = 0
            print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        else: 
            patience += 1
            if (rnd+1) % 5 == 0:
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
        if patience >= 15:
            print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)

    # FedPer head finetune
    for p in gm.encoder.parameters(): p.requires_grad_(False)
    for ft_rnd in range(cfg.head_finetune_rounds):
        for c in clients:
            opt = AdamW(gm.head.parameters(), lr=cfg.lr_min*5)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            for _ in range(5):
                gm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    lo, _ = gm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(lo[vm], lb, dev)
                amp_step(gm, opt, scaler, loss)
    for p in gm.encoder.parameters(): p.requires_grad_(True)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'
    data = load_and_build_graph(DATA_DIR).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = temporal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} IEEE-CIS External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-"*95)
    for method, runs in results.items():
        f1m,f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am,as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm,rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs])/1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")

Loading IEEE-CIS Data...
Graph: 590540 nodes | 1180046 edges | 817 feats

========================= SEED 42 =========================
  [LocalOnly]
    Client 0: F1=0.3542 AUC=0.7513
    Client 1: F1=0.3193 AUC=0.7968
    Client 2: F1=0.3385 AUC=0.8077
    Client 3: F1=0.3234 AUC=0.8054
  => F1: 0.3338, AUC: 0.7903
  [FedProto-8]
    [NEW BEST] Round   1: F1=0.2853 AUC=0.7731
    [NEW BEST] Round   2: F1=0.3670 AUC=0.8124
    [NEW BEST] Round   3: F1=0.3855 AUC=0.8207
    [NEW BEST] Round   4: F1=0.3895 AUC=0.8199
    [NEW BEST] Round   5: F1=0.3909 AUC=0.8204
    [NEW BEST] Round   6: F1=0.3971 AUC=0.8331
    [NEW BEST] Round   7: F1=0.4110 AUC=0.8385
    [NEW BEST] Round   8: F1=0.4174 AUC=0.8394
    [NEW BEST] Round   9: F1=0.4225 AUC=0.8500
    Round  10: F1=0.3877 AUC=0.8484 (best=0.4225, pat=1)
    [NEW BEST] Round  14: F1=0.4243 AUC=0.8569
    [NEW BEST] Round  15: F1=0.4278 AUC=0.8557
    [NEW BEST] Round  16: F1=0.4365 AUC=0.8564
    [NEW BEST] Round  17: F1=0.4498 AUC=0.8587
